# LeetCode 158 — Read N Characters Given read4 II - Call Multiple Times bằng Python

> **Lưu ý:** đây là bài **Premium** (bị khóa trên LeetCode, cần tài khoản trả phí mới nộp được). Notebook tự mô phỏng toàn bộ môi trường của đề (file, hàm `read4`, cách LeetCode gọi `read` nhiều lần) nên bạn vẫn học và chạy thử đầy đủ mà không cần tài khoản Premium.

Notebook này giảng nhập môn bài **Hard**:

```python
# The read4 API is already defined for you.
# def read4(buf4: List[str]) -> int:

class Solution:
    def read(self, buf: List[str], n: int) -> int:

```

Bài này **không khó về thuật toán** (không có DP, không có đồ thị). Cái "Hard" nằm ở chỗ phải **thiết kế trạng thái** đúng: hàm `read` bị gọi **nhiều lần liên tiếp**, và những ký tự "đọc dư" ở lần trước **không được phép mất**.

Mục tiêu của notebook:

- Hiểu các khái niệm nền: **file pointer** (con trỏ đọc), **buffer** (bộ đệm), "buffer là **đích** chứ không phải nguồn".
- Hiểu vì sao Python cho phép hàm **ghi vào list được truyền vào** (mutate) và vì sao `buf = ...` bên trong hàm lại vô dụng.
- Biết cách **giữ trạng thái giữa các lần gọi** bằng `self.*` trong class, và cái bẫy *class attribute* dùng chung.
- Ôn lại bản I (**#157**, chỉ gọi 1 lần), rồi thấy tận mắt vì sao code bản I **sai** khi gọi nhiều lần.
- Học 3 lời giải đúng: đọc hết file vào bộ nhớ → **bộ đệm 4 ký tự + `buf_ptr`/`buf_cnt`** (cách chính) → **hàng đợi `deque`** chứa phần dư.
- Tự kiểm tra bằng bộ mô phỏng giống LeetCode, test ngẫu nhiên và kiểm tra bất biến (invariant).

## 1. Đề bài

Cho một **file**, và bạn **chỉ được đọc file thông qua hàm `read4`** có sẵn. Hãy cài đặt hàm `read` để đọc `n` ký tự. **Hàm `read` của bạn có thể bị gọi nhiều lần.**

### Hàm `read4` (đề cho sẵn)

`read4` đọc **4 ký tự liên tiếp** từ file, rồi ghi các ký tự đó vào mảng `buf4`.

- Tham số: `buf4` — một mảng ký tự (trong Python là `List[str]`).
- Giá trị trả về: số ký tự **thật sự** đọc được (từ 0 đến 4).
- `buf4` là **đích** (destination), không phải nguồn: `read4` **ghi** kết quả vào `buf4`.
- `read4` có **con trỏ file riêng** (giống `FILE *fp` trong C): mỗi lần gọi, nó đọc tiếp từ chỗ lần trước dừng lại.

Ví dụ minh họa của đề:

```text
File file("abcde");   // File là "abcde", ban đầu con trỏ fp trỏ vào 'a'
char[] buf4 = new char[4];
read4(buf4);          // trả 4. Bây giờ buf4 = "abcd", fp trỏ vào 'e'
read4(buf4);          // trả 1. Bây giờ buf4 = "e",    fp trỏ tới cuối file
read4(buf4);          // trả 0. Bây giờ buf4 = "",     fp trỏ tới cuối file
```

### Hàm `read` (bạn phải viết)

Dùng `read4`, hãy cài đặt `read(buf, n)`: đọc **tối đa `n` ký tự** từ file và **ghi vào mảng `buf`**.

- Giá trị trả về: số ký tự thật sự đọc được (có thể ít hơn `n` nếu file đã hết).
- `buf` cũng là **đích**: bạn phải **ghi** kết quả vào `buf`.

### Ghi chú của đề (rất quan trọng)

1. Bạn **không được** thao tác trực tiếp lên file. File chỉ truy cập được qua `read4`, **không** qua `read`.
2. Hàm `read` **có thể bị gọi nhiều lần**.
3. Hãy nhớ **RESET các biến của class** khai báo trong `Solution`, vì biến static/class được **giữ lại giữa các test case**.
4. Có thể giả sử `buf` luôn **đủ chỗ** để chứa `n` ký tự.
5. Trong một test case, **cùng một `buf`** được dùng cho các lần gọi `read`.

### Ví dụ của đề

**Ví dụ 1**

```text
Input:  file = "abc", queries = [1,2,1]
Output: [1,2,0]
```

Giải thích — test case này tương đương kịch bản:

```text
File file("abc");
Solution sol;
sol.read(buf, 1);  // buf chứa "a".  Đọc được 1 ký tự  -> trả 1
sol.read(buf, 2);  // buf chứa "bc". Đọc được 2 ký tự  -> trả 2
sol.read(buf, 1);  // Đã tới cuối file, không đọc được gì -> trả 0
```

**Ví dụ 2**

```text
Input:  file = "abc", queries = [4,1]
Output: [3,0]
```

```text
File file("abc");
Solution sol;
sol.read(buf, 4);  // buf chứa "abc". File chỉ có 3 ký tự -> trả 3
sol.read(buf, 1);  // Đã tới cuối file -> trả 0
```

### Constraints

- `1 <= file.length <= 500`
- `file` gồm chữ cái tiếng Anh và chữ số.
- `1 <= queries.length <= 10`
- `1 <= queries[i] <= 500`

`queries` là danh sách giá trị `n` cho từng lần gọi `read`. Mảng `Output` là danh sách **giá trị trả về** của từng lần gọi; ngoài ra nội dung ghi vào `buf` cũng phải đúng như phần giải thích, nên các test trong notebook này kiểm tra **cả hai**.

### Hiểu đề bằng hình ảnh

Hãy tưởng tượng file là một **cuộn băng** ký tự, và `read4` là một cái máy chỉ biết **cắt đúng 4 ô một lần** (hoặc ít hơn nếu gần hết băng):

```text
file:   a  b  c  d  e  f  g
        ^
        fp (con trỏ file của read4)

read4 lần 1 -> lấy "abcd", fp nhảy tới 'e'
read4 lần 2 -> lấy "efg",  fp tới cuối
read4 lần 3 -> lấy "",     fp vẫn ở cuối
```

Còn người gọi `read` thì **muốn số lượng tùy ý**: lúc thì 1 ký tự, lúc 2, lúc 5...

Mâu thuẫn chính của bài:

- `read4` **luôn cắt theo khối 4**.
- `read(n)` muốn **đúng `n`** ký tự.
- Khi `n` không phải bội của 4, ta sẽ **cầm dư** vài ký tự. Lần gọi `read` **sau** phải nhận được đúng những ký tự dư đó trước tiên, vì `read4` **không thể quay lui** con trỏ file.

Chỉ cần nhớ câu này: **"cắt dư thì phải cất đi, lần sau lấy ra dùng trước."** Toàn bộ bài là cài đặt câu đó cho đúng.

## 2. Kiến thức nền tảng

### 2.1. File pointer (con trỏ đọc) — tự mô phỏng một file

**File pointer** là một con số ghi nhớ "đã đọc tới vị trí nào". Mỗi lần đọc, con trỏ tiến lên; nó **không bao giờ lùi lại**.

Ta tự viết một class `File` nhỏ để mô phỏng:

- `content`: nội dung file (một chuỗi).
- `pos`: con trỏ đọc, ban đầu bằng `0` (trỏ vào ký tự đầu tiên).
- `read4(buf4)`: chép tối đa 4 ký tự từ `content[pos]` trở đi vào `buf4`, tăng `pos`, trả về số ký tự đã chép.
- `read4_calls`: đếm số lần `read4` bị gọi (để lát nữa so sánh chi phí các cách giải).

In [1]:
from typing import List


class File:
    # Mô phỏng một file chỉ đọc được qua read4. pos = con trỏ file (file pointer).
    def __init__(self, content: str):
        self.content = content
        self.pos = 0
        self.read4_calls = 0

    def read4(self, buf4: List[str]) -> int:
        self.read4_calls += 1
        count = 0
        while count < 4 and self.pos < len(self.content):
            buf4[count] = self.content[self.pos]   # GHI vào buf4 (buf4 là đích)
            self.pos += 1
            count += 1
        return count


f = File("abcde")
buf4 = [''] * 4            # tạo sẵn 4 ô trống để read4 ghi vào
print(f.read4(buf4), buf4, "pos =", f.pos)
print(f.read4(buf4), buf4, "pos =", f.pos)
print(f.read4(buf4), buf4, "pos =", f.pos)

4 ['a', 'b', 'c', 'd'] pos = 4
1 ['e', 'b', 'c', 'd'] pos = 5
0 ['e', 'b', 'c', 'd'] pos = 5


Kết quả khớp với ví dụ của đề: trả `4`, rồi `1`, rồi `0`.

Nhưng hãy nhìn kỹ dòng thứ hai: `read4` trả `1` mà `buf4` lại là `['e', 'b', 'c', 'd']`!

Vì sao? `read4` chỉ **ghi đè** ô `buf4[0] = 'e'`. Ba ô sau vẫn còn **rác cũ** từ lần đọc trước. Đề viết "buf4 = "e"" nghĩa là **chỉ `count` ô đầu là có nghĩa**.

**Quy tắc vàng:** sau `cnt = read4(buf4)`, chỉ được dùng `buf4[0]`, ..., `buf4[cnt - 1]`. Đừng bao giờ dùng cả 4 ô.

In [2]:
f = File("abcde")
buf4 = [''] * 4
f.read4(buf4)
cnt = f.read4(buf4)
print("cnt =", cnt)
print("toàn bộ buf4     :", buf4)          # có rác 'b','c','d'
print("phần có nghĩa    :", buf4[:cnt])    # chỉ ['e']

cnt = 1
toàn bộ buf4     : ['e', 'b', 'c', 'd']
phần có nghĩa    : ['e']


### 2.2. Buffer là "đích": hàm ghi vào list được truyền vào

Đề nói "`buf` là đích, không phải nguồn". Với Python, điều này có nghĩa:

- Người gọi tạo sẵn một list, ví dụ `buf = [''] * 5`.
- Người gọi đưa list đó vào hàm: `read(buf, 5)`.
- Hàm `read` phải **sửa từng ô** của chính list đó: `buf[0] = 'a'`, `buf[1] = 'b'`, ...
- Sau khi hàm trả về, người gọi nhìn vào **cùng list đó** và thấy dữ liệu mới.

Điều này chạy được vì trong Python, truyền một list vào hàm là truyền **tham chiếu tới cùng một object** — hàm và người gọi cùng "cầm" một cái hộp. Sửa ô bên trong hộp (**mutate**) thì cả hai đều thấy.

Nhưng nếu trong hàm viết `buf = ['x', 'y']` thì đó là **gán lại tên** (rebind): tên `buf` bên trong hàm giờ trỏ sang một hộp **mới**, còn hộp của người gọi vẫn y nguyên. Đây là lỗi người mới rất hay mắc.

Một cách ghi nhiều ô cùng lúc mà vẫn là **mutate** là **gán slice** (slice assignment):

```python
buf[i:i + k] = some_list_of_length_k
```

Nếu vế phải có **đúng** `k` phần tử thì list giữ nguyên độ dài, chỉ các ô `i..i+k-1` bị thay. Ta sẽ dùng mẹo này ở biến thể "chép theo khối".

In [3]:
def fill_mutate(buf):
    buf[0] = 'x'          # sửa ô trong CÙNG list -> người gọi thấy
    buf[1] = 'y'

def fill_rebind(buf):
    buf = ['x', 'y']      # tạo list MỚI, chỉ đổi tên buf cục bộ -> người gọi KHÔNG thấy

a = ['', '', '']
fill_mutate(a)
print("sau fill_mutate:", a)

b = ['', '', '']
fill_rebind(b)
print("sau fill_rebind:", b)

buf = ['.'] * 6
buf[1:4] = ['a', 'b', 'c']     # thay 3 ô từ chỉ số 1 tới 3
print(buf, "độ dài =", len(buf))

sau fill_mutate: ['x', 'y', '']
sau fill_rebind: ['', '', '']
['.', 'a', 'b', 'c', '.', '.'] độ dài = 6


### 2.3. Trạng thái giữa các lần gọi: biến cục bộ "quên", `self.*` "nhớ"

Biến tạo bên trong một hàm là **biến cục bộ** (local variable): mỗi lần gọi hàm, nó được tạo mới và **biến mất** khi hàm kết thúc. Hàm **không nhớ** gì từ lần gọi trước.

Muốn nhớ, ta đặt dữ liệu vào **object**: trong class, `self.ten_bien` là **thuộc tính của object** (instance attribute). Nó sống cùng object, nên lần gọi method sau vẫn thấy giá trị lần trước để lại.

Ví dụ mini: đếm số lần được gọi.

In [4]:
def counter_function():
    count = 0          # tạo mới mỗi lần gọi
    count += 1
    return count

print("hàm thường:", counter_function(), counter_function(), counter_function())


class Counter:
    def __init__(self):
        self.count = 0  # sống cùng object

    def hit(self):
        self.count += 1
        return self.count

c = Counter()
print("dùng self :", c.hit(), c.hit(), c.hit())

c2 = Counter()          # object mới -> trạng thái mới, độc lập với c
print("object mới:", c2.hit())

hàm thường: 1 1 1
dùng self : 1 2 3
object mới: 1


Đây chính là chìa khóa của bài: LeetCode tạo **một** object `sol = Solution()` cho mỗi test case rồi gọi `sol.read(...)` nhiều lần. Những ký tự đọc dư phải được cất vào `self.*` để lần gọi sau lấy ra.

### 2.4. Bẫy: *class attribute* bị dùng chung

Nếu khai báo biến **ngay dưới dòng `class`** (không qua `self` trong `__init__`), đó là **class attribute** — thuộc về **class**, mọi object dùng **chung**. Với kiểu dữ liệu sửa được (list, deque, dict), sửa từ object này thì object kia cũng thấy.

Ghi chú "Please remember to RESET your class variables" của đề chính là cảnh báo bẫy này: LeetCode chạy nhiều test case trong **cùng một tiến trình**, nên class attribute của test trước còn "dính" sang test sau.

In [5]:
class BadBox:
    items = []                  # class attribute: CẢ CLASS dùng chung 1 list

    def add(self, x):
        self.items.append(x)


class GoodBox:
    def __init__(self):
        self.items = []         # instance attribute: mỗi object 1 list riêng

    def add(self, x):
        self.items.append(x)


b1, b2 = BadBox(), BadBox()
b1.add('a')
print("BadBox : b2.items =", b2.items, "(b2 chưa add gì mà đã có 'a'!)")

g1, g2 = GoodBox(), GoodBox()
g1.add('a')
print("GoodBox: g2.items =", g2.items)

BadBox : b2.items = ['a'] (b2 chưa add gì mà đã có 'a'!)
GoodBox: g2.items = []


**Quy tắc:** mọi trạng thái của `Solution` đều khởi tạo trong `__init__` bằng `self.xxx = ...`. Như vậy mỗi test case (mỗi `Solution()` mới) tự động có trạng thái sạch.

### 2.5. Mô phỏng cách LeetCode gọi `read4` và `read`

Trên LeetCode, `read4` là một **hàm toàn cục** (global function) đã được định nghĩa sẵn và "gắn" với file của test case hiện tại. Code của bạn chỉ việc gọi `read4(buf4)`.

Ta mô phỏng y hệt:

- `set_file(content)`: tạo file mới cho một test case, lưu vào biến toàn cục `_current_file`.
- `read4(buf4)`: hàm toàn cục, gọi sang `_current_file.read4(buf4)`.
- `run_queries(SolutionClass, content, queries)`: làm đúng như LeetCode — tạo file, tạo **một** object `Solution`, rồi với mỗi `n` trong `queries` thì tạo `buf` đủ `n` ô và gọi `sol.read(buf, n)`. Trả về danh sách giá trị trả về và nội dung đọc được.
- `expected_output(content, queries)`: "đáp án chuẩn" tính bằng cách cắt chuỗi trực tiếp — **chỉ dùng để kiểm tra**, vì lời giải thật không được chạm vào file.

Nhờ vậy, mọi class `Solution` trong notebook đều có thể **copy nguyên văn** lên LeetCode.

In [6]:
_current_file = None


def set_file(content: str) -> File:
    global _current_file
    _current_file = File(content)
    return _current_file


def read4(buf4: List[str]) -> int:
    # Giống LeetCode: read4 là hàm toàn cục, đọc từ file của test case hiện tại.
    return _current_file.read4(buf4)


def run_queries(solution_cls, content, queries, show=False):
    set_file(content)
    sol = solution_cls()                 # MỘT object cho cả test case
    counts, texts = [], []
    for n in queries:
        buf = [''] * n                   # buf đủ chỗ cho n ký tự
        cnt = sol.read(buf, n)
        counts.append(cnt)
        texts.append(''.join(buf[:cnt]))
        if show:
            print(f"read(buf, {n}) -> {cnt}, buf chứa {''.join(buf[:cnt])!r}")
    return counts, texts


def expected_output(content, queries):
    # Đáp án chuẩn: cắt chuỗi trực tiếp (chỉ để kiểm tra, lời giải thật không được làm vậy).
    pos, counts, texts = 0, [], []
    for n in queries:
        piece = content[pos:pos + n]
        pos += len(piece)
        counts.append(len(piece))
        texts.append(piece)
    return counts, texts


def check_queries(solution_cls, content, queries):
    actual = run_queries(solution_cls, content, queries)
    expected = expected_output(content, queries)
    status = "PASS" if actual == expected else "FAIL"
    print(f"{status} | file={content!r} queries={queries} -> {actual[0]} {actual[1]}"
          + ("" if status == "PASS" else f" | mong đợi {expected[0]} {expected[1]}"))
    return actual == expected


print(expected_output("abc", [1, 2, 1]))
print(expected_output("abc", [4, 1]))

([1, 2, 0], ['a', 'bc', ''])
([3, 0], ['abc', ''])


### 2.6. `min()` — "lấy bao nhiêu ký tự là vừa?"

Khi `read4` trả về `cnt` ký tự mà ta chỉ còn cần `need` ký tự, số ký tự nên lấy là:

```python
take = min(cnt, need)
```

- `cnt = 4`, `need = 1` → lấy 1 (dư 3).
- `cnt = 2`, `need = 5` → lấy 2 (file sắp hết, lấy hết những gì có).
- `cnt = 0` → lấy 0, nghĩa là **file đã hết** (EOF — End Of File), phải dừng.

In [7]:
for cnt, need in [(4, 1), (4, 4), (2, 5), (0, 3)]:
    take = min(cnt, need)
    print(f"cnt={cnt}, need={need} -> take={take}, dư={cnt - take}")

cnt=4, need=1 -> take=1, dư=3
cnt=4, need=4 -> take=4, dư=0
cnt=2, need=5 -> take=2, dư=0
cnt=0, need=3 -> take=0, dư=0


## 3. Ôn bản I — LeetCode #157 (chỉ gọi `read` **một lần**)

Bài **#157 Read N Characters Given Read4** giống hệt đề này, chỉ khác: `read` **chỉ được gọi đúng một lần**.

Ý tưởng tự nhiên:

1. Tạo `buf4` 4 ô và biến `total = 0` (đã ghi bao nhiêu ký tự vào `buf`).
2. Lặp khi `total < n`:
   - Gọi `cnt = read4(buf4)`. Nếu `cnt == 0` → file hết, dừng.
   - Lấy `take = min(cnt, n - total)` ký tự đầu của `buf4`, chép vào `buf[total], buf[total+1], ...`.
   - `total += take`.
3. Trả về `total`.

Phần dư `cnt - take` bị **bỏ đi** — với bản I điều đó vô hại, vì không có lần gọi sau.

In [8]:
def read_once(buf: List[str], n: int) -> int:
    # Lời giải bản I (#157): chỉ đúng nếu được gọi MỘT lần.
    buf4 = [''] * 4
    total = 0
    while total < n:
        cnt = read4(buf4)
        if cnt == 0:                      # hết file
            break
        take = min(cnt, n - total)
        for i in range(take):
            buf[total + i] = buf4[i]
        total += take                     # phần dư (cnt - take) bị bỏ đi
    return total


for content, n in [("abc", 4), ("abcde", 5), ("leetcode", 5)]:
    set_file(content)
    buf = [''] * n
    cnt = read_once(buf, n)
    print(f"file={content!r}, n={n} -> trả {cnt}, buf = {''.join(buf[:cnt])!r}")

file='abc', n=4 -> trả 3, buf = 'abc'
file='abcde', n=5 -> trả 5, buf = 'abcde'
file='leetcode', n=5 -> trả 5, buf = 'leetc'


### Trace bản I với `file = "leetcode"`, `n = 5`

| Lần gọi read4 | `cnt` | `buf4` có nghĩa | `need = n - total` | `take` | `total` sau | Bỏ đi |
|---:|---:|---|---:|---:|---:|---|
| 1 | 4 | `l e e t` | 5 | 4 | 4 | (không) |
| 2 | 4 | `c o d e` | 1 | 1 | 5 | `o d e` |

Vòng lặp dừng vì `total == n == 5`. Kết quả `"leetc"`. Ba ký tự `"ode"` bị vứt — ghi nhớ chi tiết này, lát nữa nó chính là thủ phạm.

In [9]:
def trace_read_once(content: str, n: int) -> None:
    set_file(content)
    buf, buf4, total, call = [''] * n, [''] * 4, 0, 0
    while total < n:
        cnt = read4(buf4)
        call += 1
        if cnt == 0:
            print(f"read4 lần {call}: cnt=0 -> hết file, dừng")
            break
        take = min(cnt, n - total)
        for i in range(take):
            buf[total + i] = buf4[i]
        print(f"read4 lần {call}: cnt={cnt}, buf4={buf4[:cnt]}, need={n - total}, "
              f"take={take}, bỏ đi={buf4[take:cnt]}")
        total += take
    print(f"=> trả {total}, buf = {''.join(buf[:total])!r}")

trace_read_once("leetcode", 5)

read4 lần 1: cnt=4, buf4=['l', 'e', 'e', 't'], need=5, take=4, bỏ đi=[]
read4 lần 2: cnt=4, buf4=['c', 'o', 'd', 'e'], need=1, take=1, bỏ đi=['o', 'd', 'e']
=> trả 5, buf = 'leetc'


**Độ phức tạp của bản I** (gọi đúng 1 lần, `n` = số ký tự yêu cầu):

- Thời gian: `O(n)` — gọi `read4` khoảng `n / 4 + 1` lần, mỗi ký tự chép một lần.
- Bộ nhớ: `O(1)` — chỉ có `buf4` 4 ô.

Bản I hoàn toàn đúng cho #157. Câu hỏi của bản II: **điều gì xảy ra nếu gọi nó nhiều lần?**

## 4. Cách 1 — Ngây thơ: dùng nguyên code bản I cho mỗi lần gọi (SAI)

Người mới thường nghĩ: "Bản II chỉ là gọi bản I nhiều lần thôi mà." Ta bọc `read_once` vào class:

```python
class SolutionNaive:
    def read(self, buf, n):
        return read_once(buf, n)
```

Thử với **Ví dụ 1** của đề: `file = "abc"`, `queries = [1, 2, 1]`, mong đợi `[1, 2, 0]`.

In [10]:
class SolutionNaive:
    def read(self, buf: List[str], n: int) -> int:
        return read_once(buf, n)      # mỗi lần gọi đều bắt đầu lại từ đầu, không nhớ gì


run_queries(SolutionNaive, "abc", [1, 2, 1], show=True)
print()
check_queries(SolutionNaive, "abc", [1, 2, 1])
check_queries(SolutionNaive, "abc", [4, 1])

read(buf, 1) -> 1, buf chứa 'a'
read(buf, 2) -> 0, buf chứa ''
read(buf, 1) -> 0, buf chứa ''

FAIL | file='abc' queries=[1, 2, 1] -> [1, 0, 0] ['a', '', ''] | mong đợi [1, 2, 0] ['a', 'bc', '']
PASS | file='abc' queries=[4, 1] -> [3, 0] ['abc', '']


True

**Sai ở Ví dụ 1!** Ra `[1, 0, 0]` thay vì `[1, 2, 0]`. (Ví dụ 2 tình cờ đúng vì lần đầu đã đọc hết file.)

Chuyện gì đã xảy ra?

```text
read(buf, 1):
    read4 -> "abc" (cnt = 3), con trỏ file nhảy tới CUỐI FILE
    lấy 'a', vứt "bc"          <-- dữ liệu bị mất ở đây
read(buf, 2):
    read4 -> "" (cnt = 0), vì con trỏ file đã ở cuối
    trả 0                      <-- lẽ ra phải là "bc"
```

Mấu chốt: `read4` có con trỏ **riêng** và con trỏ đó **không lùi lại được**. Khi ta đã gọi `read4` lấy `"abc"`, thì `"bc"` chỉ còn tồn tại **trong tay ta** (trong `buf4`). Vứt `buf4` (vì nó là biến cục bộ) là mất vĩnh viễn.

Trace dưới đây in rõ ký tự bị vứt ở mỗi lần gọi.

In [11]:
def trace_naive(content, queries):
    set_file(content)
    for n in queries:
        buf4, total, got, dropped = [''] * 4, 0, [], []
        while total < n:
            cnt = read4(buf4)
            if cnt == 0:
                break
            take = min(cnt, n - total)
            got += buf4[:take]
            dropped += buf4[take:cnt]
            total += take
        print(f"read({n}): trả {total}, nhận {''.join(got)!r}, VỨT {''.join(dropped)!r}, "
              f"con trỏ file = {_current_file.pos}")

trace_naive("abc", [1, 2, 1])
print()
trace_naive("abcdefgh", [2, 2, 4])

read(1): trả 1, nhận 'a', VỨT 'bc', con trỏ file = 3
read(2): trả 0, nhận '', VỨT '', con trỏ file = 3
read(1): trả 0, nhận '', VỨT '', con trỏ file = 3

read(2): trả 2, nhận 'ab', VỨT 'cd', con trỏ file = 4
read(2): trả 2, nhận 'ef', VỨT 'gh', con trỏ file = 8
read(4): trả 0, nhận '', VỨT '', con trỏ file = 8


Ví dụ thứ hai còn rõ hơn: file `"abcdefgh"`, gọi `read(2)`, `read(2)`, `read(4)`. Đáp án đúng là `"ab"`, `"cd"`, `"efgh"`; cách ngây thơ cho `"ab"`, `"ef"`, `""` vì `"cd"` và `"gh"` bị vứt.

**Bài học (bước đột phá đầu tiên):** những ký tự `read4` đã trả về mà `read` chưa dùng tới **phải được cất lại trong object** (`self.*`), và lần gọi `read` sau phải **giao chúng trước** rồi mới gọi `read4` tiếp.

Câu hỏi tiếp theo: cất **bao nhiêu** và cất **thế nào**?

## 5. Cách 2 — Đọc hết file vào một bộ đệm lớn ngay lần đầu (đúng nhưng chưa tốt)

Ý tưởng "an toàn tuyệt đối": nếu sợ mất dữ liệu thì ở **lần gọi đầu tiên**, cứ gọi `read4` liên tục cho tới khi nó trả `0`, gom **toàn bộ file** vào `self.data`. Sau đó mỗi lần `read(n)` chỉ việc lấy tiếp từ `self.data` bằng con trỏ của riêng mình `self.pos`.

Trạng thái cần nhớ:

- `self.data`: list toàn bộ ký tự của file (`None` nếu chưa đọc).
- `self.pos`: đã giao cho người gọi tới vị trí nào trong `self.data`.

Ở đây ta có **hai con trỏ** khác nhau, đừng nhầm:

- Con trỏ file của `read4` (ta không điều khiển được).
- `self.pos` — con trỏ **của ta** trên bản sao trong bộ nhớ.

In [12]:
class SolutionReadAll:
    def __init__(self):
        self.data = None       # chưa đọc file
        self.pos = 0           # vị trí kế tiếp sẽ giao trong self.data

    def read(self, buf: List[str], n: int) -> int:
        if self.data is None:                 # lần đầu: hút hết file
            self.data = []
            buf4 = [''] * 4
            while True:
                cnt = read4(buf4)
                if cnt == 0:
                    break
                self.data.extend(buf4[:cnt])  # chỉ lấy cnt ô có nghĩa
        count = 0
        while count < n and self.pos < len(self.data):
            buf[count] = self.data[self.pos]
            count += 1
            self.pos += 1
        return count


run_queries(SolutionReadAll, "abc", [1, 2, 1], show=True)
print()
check_queries(SolutionReadAll, "abc", [1, 2, 1])
check_queries(SolutionReadAll, "abc", [4, 1])
check_queries(SolutionReadAll, "abcdefgh", [2, 2, 4])

read(buf, 1) -> 1, buf chứa 'a'
read(buf, 2) -> 2, buf chứa 'bc'
read(buf, 1) -> 0, buf chứa ''

PASS | file='abc' queries=[1, 2, 1] -> [1, 2, 0] ['a', 'bc', '']
PASS | file='abc' queries=[4, 1] -> [3, 0] ['abc', '']
PASS | file='abcdefgh' queries=[2, 2, 4] -> [2, 2, 4] ['ab', 'cd', 'efgh']


True

Đúng cả ba. Vậy vấn đề là gì? Hãy đếm xem **chỉ để đọc 1 ký tự**, cách này gọi `read4` bao nhiêu lần và giữ bao nhiêu ký tự trong bộ nhớ, với file dài 500 ký tự (giới hạn của đề).

In [13]:
big = "x" * 500
run_queries(SolutionReadAll, big, [1])
print("Cách 2, read(1) trên file 500 ký tự: read4 bị gọi", _current_file.read4_calls, "lần")

run_queries(SolutionNaive, big, [1])          # để so: chỉ cần 1 lần read4 là đủ 1 ký tự
print("Chỉ để lấy 1 ký tự, tối thiểu cần   :", _current_file.read4_calls, "lần")

Cách 2, read(1) trên file 500 ký tự: read4 bị gọi 126 lần
Chỉ để lấy 1 ký tự, tối thiểu cần   : 1 lần


`126` lần gọi `read4` (125 lần có dữ liệu + 1 lần trả 0) và giữ cả 500 ký tự trong bộ nhớ, chỉ để trả về **1** ký tự.

**Độ phức tạp cách 2** (`L` = độ dài file, `n` = số ký tự yêu cầu ở một lần gọi):

- Lần gọi đầu: thời gian `O(L)` (đọc cả file), các lần sau `O(n)`.
- Bộ nhớ: `O(L)`.

Với `L <= 500`, LeetCode vẫn chấp nhận. Nhưng nó **đi ngược tinh thần của bài**:

- `read4` mô phỏng việc đọc **luồng dữ liệu** (stream): file 10 GB, dữ liệu mạng, dữ liệu từ bàn phím... Không thể (hoặc không nên) hút hết vào RAM.
- Có những stream **không có điểm kết thúc** (ví dụ socket mạng còn mở): cách 2 sẽ đứng chờ mãi ở lần gọi đầu.
- Người phỏng vấn gần như chắc chắn hỏi: "Nếu file rất lớn thì sao?"

**Nút thắt:** ta giữ **quá nhiều**. Thực ra ta chỉ cần giữ **phần dư** của **lần `read4` gần nhất**. Phần dư đó lớn tới đâu?

## 6. Bước đột phá — phần dư không bao giờ quá 3 ký tự

Quan sát:

- Mỗi lần `read4` trả về **tối đa 4** ký tự.
- Ta chỉ gọi `read4` khi **cần thêm ít nhất 1 ký tự** và **không còn gì dư** trong tay.
- Vậy sau khi gọi, ta dùng **ít nhất 1** ký tự ngay → phần dư tối đa `4 - 1 = 3` ký tự.

Nghĩa là chỉ cần một **bộ đệm nội bộ 4 ô** `self.buf4` cộng hai con số:

- `self.buf_cnt`: lần `read4` gần nhất trả về bao nhiêu ký tự (bao nhiêu ô của `self.buf4` có nghĩa).
- `self.buf_ptr`: ô tiếp theo **chưa giao** cho người gọi.

```text
self.buf4:   [ a | b | c | d ]
               0   1   2   3
                   ^           ^
               buf_ptr = 1   buf_cnt = 4

đã giao:  'a'
còn dư:   buf4[buf_ptr : buf_cnt] = "bcd"
```

Hai trạng thái đặc biệt:

- `buf_ptr == buf_cnt` → **bộ đệm rỗng**, cần gọi `read4` để nạp lại (refill).
- Sau khi nạp lại mà `buf_cnt == 0` → **file hết**, dừng.

Ban đầu `buf_ptr = buf_cnt = 0` nghĩa là "rỗng", nên lần đầu tự động nạp. Không cần trường hợp riêng cho lần gọi đầu như cách 2.

### Thí nghiệm nhỏ: một "máy phát từng ký tự" dùng bộ đệm 4 ô

Trước khi viết `read(n)`, ta viết thứ đơn giản hơn: `next_char()` trả về **một** ký tự kế tiếp của file (hoặc `''` nếu hết), dùng đúng ba biến trạng thái ở trên. Nếu làm được `next_char()`, thì `read(n)` chỉ là gọi `next_char()` tối đa `n` lần.

In [14]:
class CharReader:
    def __init__(self):
        self.buf4 = [''] * 4
        self.buf_ptr = 0
        self.buf_cnt = 0

    def next_char(self) -> str:
        if self.buf_ptr == self.buf_cnt:          # bộ đệm rỗng -> nạp lại
            self.buf_cnt = read4(self.buf4)
            self.buf_ptr = 0
            print(f"    (nạp lại: read4 trả {self.buf_cnt} -> {self.buf4[:self.buf_cnt]})")
            if self.buf_cnt == 0:
                return ''                         # hết file
        ch = self.buf4[self.buf_ptr]
        self.buf_ptr += 1
        return ch


set_file("abcdef")
r = CharReader()
for _ in range(7):
    ch = r.next_char()
    print(f"next_char() -> {ch!r:4} | buf_ptr={r.buf_ptr}, buf_cnt={r.buf_cnt}")

    (nạp lại: read4 trả 4 -> ['a', 'b', 'c', 'd'])
next_char() -> 'a'  | buf_ptr=1, buf_cnt=4
next_char() -> 'b'  | buf_ptr=2, buf_cnt=4
next_char() -> 'c'  | buf_ptr=3, buf_cnt=4
next_char() -> 'd'  | buf_ptr=4, buf_cnt=4
    (nạp lại: read4 trả 2 -> ['e', 'f'])
next_char() -> 'e'  | buf_ptr=1, buf_cnt=2
next_char() -> 'f'  | buf_ptr=2, buf_cnt=2
    (nạp lại: read4 trả 0 -> [])
next_char() -> ''   | buf_ptr=0, buf_cnt=0


Quan sát output:

- Lần gọi đầu tiên phải nạp, lấy được `"abcd"` và trả `'a'`. Ba lần `next_char()` tiếp theo (`'b'`, `'c'`, `'d'`) **không gọi `read4`** — chúng chỉ tăng `buf_ptr` từ 1 lên 4.
- Khi `buf_ptr == buf_cnt == 4` → nạp lại, được `"ef"` (`buf_cnt = 2`).
- Khi `"ef"` dùng hết, lần gọi thứ 7 nạp lại, `read4` trả `0` → trả `''` (hết file). Gọi thêm nữa cũng chỉ nhận `''`.

Không ký tự nào bị vứt, và bộ nhớ dùng thêm chỉ là 4 ô. Đây là toàn bộ "linh hồn" của lời giải chính.

## 7. Cách 3 — Bộ đệm nội bộ 4 ô + `buf_ptr` / `buf_cnt` (cách chính)

Ghép `next_char()` vào vòng lặp của `read(buf, n)`:

**Bước 1 — Khởi tạo trạng thái trong `__init__`** (không dùng class attribute!):

```python
self.buf4 = [''] * 4
self.buf_ptr = 0
self.buf_cnt = 0
```

**Bước 2 — Vòng lặp:** `idx` là số ký tự đã ghi vào `buf` trong lần gọi này. Lặp khi `idx < n`.

**Bước 3 — Nếu bộ đệm rỗng (`buf_ptr == buf_cnt`):** gọi `read4(self.buf4)`, gán `buf_cnt`, **đặt lại `buf_ptr = 0`**. Nếu `buf_cnt == 0` → file hết → `break`.

**Bước 4 — Chép 1 ký tự:** `buf[idx] = self.buf4[self.buf_ptr]`, tăng cả `idx` và `buf_ptr`.

**Bước 5 — Trả về `idx`.**

Điểm hay: khi vòng lặp dừng vì `idx == n`, những ký tự còn lại trong `self.buf4[buf_ptr:buf_cnt]` **vẫn nằm yên trong object**, chờ lần gọi sau.

In [15]:
class SolutionBuffer:
    def __init__(self):
        self.buf4 = [''] * 4     # bộ đệm nội bộ, giữ phần dư giữa các lần gọi
        self.buf_ptr = 0         # ô kế tiếp chưa giao
        self.buf_cnt = 0         # số ô có nghĩa trong buf4

    def read(self, buf: List[str], n: int) -> int:
        idx = 0
        while idx < n:
            if self.buf_ptr == self.buf_cnt:          # bộ đệm rỗng -> nạp lại
                self.buf_cnt = read4(self.buf4)
                self.buf_ptr = 0
                if self.buf_cnt == 0:                 # hết file
                    break
            buf[idx] = self.buf4[self.buf_ptr]
            idx += 1
            self.buf_ptr += 1
        return idx


run_queries(SolutionBuffer, "abc", [1, 2, 1], show=True)
print()
check_queries(SolutionBuffer, "abc", [1, 2, 1])
check_queries(SolutionBuffer, "abc", [4, 1])
check_queries(SolutionBuffer, "abcdefgh", [2, 2, 4])

read(buf, 1) -> 1, buf chứa 'a'
read(buf, 2) -> 2, buf chứa 'bc'
read(buf, 1) -> 0, buf chứa ''

PASS | file='abc' queries=[1, 2, 1] -> [1, 2, 0] ['a', 'bc', '']
PASS | file='abc' queries=[4, 1] -> [3, 0] ['abc', '']
PASS | file='abcdefgh' queries=[2, 2, 4] -> [2, 2, 4] ['ab', 'cd', 'efgh']


True

### Trace từng bước: `file = "abcdefg"`, `queries = [1, 2, 1, 5, 3]`

Chọn input này vì nó có đủ các tình huống: lần đầu nạp, giao từ phần dư mà **không** gọi `read4`, dùng hết phần dư rồi nạp lại, **hết file giữa chừng** một lần gọi, và gọi sau khi file đã hết.

Đáp án mong đợi: `"a"`, `"bc"`, `"d"`, `"efg"`, `""` → `[1, 2, 1, 3, 0]`.

| Lệnh | Còn dư trước (`buf4[ptr:cnt]`) | Gọi `read4`? | Ký tự giao | Trả về | Còn dư sau |
|---|---|---|---|---:|---|
| `read(1)` | `""` (ptr=0, cnt=0) | có → `"abcd"` | `a` | 1 | `"bcd"` |
| `read(2)` | `"bcd"` | **không** | `b c` | 2 | `"d"` |
| `read(1)` | `"d"` | **không** | `d` | 1 | `""` |
| `read(5)` | `""` | có → `"efg"`, rồi có → `""` (hết file) | `e f g` | 3 | `""` |
| `read(3)` | `""` | có → `""` (hết file) | (không) | 0 | `""` |

Tổng cộng chỉ 4 lần gọi `read4`. Ý nghĩa các cột:

- **Còn dư trước**: những ký tự đã nằm trong tay ta từ lần gọi trước — phải giao **trước tiên**.
- **Gọi `read4`?**: chỉ xảy ra khi phần dư đã hết mà vẫn còn cần ký tự.
- **Còn dư sau**: những gì để dành cho lần gọi kế tiếp.

Hàm `trace_buffer` dưới đây in đúng các quyết định đó.

In [16]:
def trace_buffer(content, queries):
    set_file(content)
    sol = SolutionBuffer()
    for n in queries:
        left = ''.join(sol.buf4[sol.buf_ptr:sol.buf_cnt])
        print(f"=== read(buf, {n}) | còn dư trước: {left!r}")
        buf, idx = [''] * n, 0
        while idx < n:
            if sol.buf_ptr == sol.buf_cnt:
                sol.buf_cnt = read4(sol.buf4)
                sol.buf_ptr = 0
                print(f"    bộ đệm rỗng -> read4 trả {sol.buf_cnt}: {sol.buf4[:sol.buf_cnt]}")
                if sol.buf_cnt == 0:
                    print("    hết file -> break")
                    break
            buf[idx] = sol.buf4[sol.buf_ptr]
            print(f"    giao buf4[{sol.buf_ptr}]={sol.buf4[sol.buf_ptr]!r} -> buf[{idx}]")
            idx += 1
            sol.buf_ptr += 1
        left = ''.join(sol.buf4[sol.buf_ptr:sol.buf_cnt])
        print(f"    => trả {idx}, buf = {''.join(buf[:idx])!r}, còn dư sau: {left!r}")
    print("Tổng số lần gọi read4:", _current_file.read4_calls)

trace_buffer("abcdefg", [1, 2, 1, 5, 3])

=== read(buf, 1) | còn dư trước: ''
    bộ đệm rỗng -> read4 trả 4: ['a', 'b', 'c', 'd']
    giao buf4[0]='a' -> buf[0]
    => trả 1, buf = 'a', còn dư sau: 'bcd'
=== read(buf, 2) | còn dư trước: 'bcd'
    giao buf4[1]='b' -> buf[0]
    giao buf4[2]='c' -> buf[1]
    => trả 2, buf = 'bc', còn dư sau: 'd'
=== read(buf, 1) | còn dư trước: 'd'
    giao buf4[3]='d' -> buf[0]
    => trả 1, buf = 'd', còn dư sau: ''
=== read(buf, 5) | còn dư trước: ''
    bộ đệm rỗng -> read4 trả 3: ['e', 'f', 'g']
    giao buf4[0]='e' -> buf[0]
    giao buf4[1]='f' -> buf[1]
    giao buf4[2]='g' -> buf[2]
    bộ đệm rỗng -> read4 trả 0: []
    hết file -> break
    => trả 3, buf = 'efg', còn dư sau: ''
=== read(buf, 3) | còn dư trước: ''
    bộ đệm rỗng -> read4 trả 0: []
    hết file -> break
    => trả 0, buf = '', còn dư sau: ''
Tổng số lần gọi read4: 4


Output khớp bảng. Ba điểm cần để ý:

1. Ở `read(2)` và `read(1)` thứ hai **không có dòng "bộ đệm rỗng"** — ký tự được giao thẳng từ phần dư. Đây chính là chỗ cách ngây thơ đã làm mất dữ liệu.
2. Ở `read(5)`, sau khi giao `"efg"` vẫn cần thêm 2 ký tự nên gọi `read4` lần nữa, nhận `0` → `break` → trả `3` (ít hơn `n` là bình thường).
3. Ở `read(3)` cuối, `read4` lại trả `0` ngay. `read4` an toàn khi gọi lặp lại ở cuối file — nó chỉ trả `0`.

### Biến thể 3b — chép theo khối thay vì từng ký tự

Thay vì chép 1 ký tự mỗi vòng, ta có thể chép **cả khối** `take = min(n - idx, buf_cnt - buf_ptr)` ký tự bằng gán slice (mục 2.2). Logic y hệt, chỉ ít vòng lặp hơn. Đây là cách viết thường gặp trong C/Java với `System.arraycopy` / `memcpy`.

In [17]:
class SolutionBufferChunk:
    def __init__(self):
        self.buf4 = [''] * 4
        self.buf_ptr = 0
        self.buf_cnt = 0

    def read(self, buf: List[str], n: int) -> int:
        idx = 0
        while idx < n:
            if self.buf_ptr == self.buf_cnt:
                self.buf_cnt = read4(self.buf4)
                self.buf_ptr = 0
                if self.buf_cnt == 0:
                    break
            take = min(n - idx, self.buf_cnt - self.buf_ptr)   # cần bao nhiêu, còn bao nhiêu
            buf[idx:idx + take] = self.buf4[self.buf_ptr:self.buf_ptr + take]
            idx += take
            self.buf_ptr += take
        return idx


check_queries(SolutionBufferChunk, "abc", [1, 2, 1])
check_queries(SolutionBufferChunk, "abc", [4, 1])
check_queries(SolutionBufferChunk, "abcdefg", [1, 2, 1, 5, 3])

PASS | file='abc' queries=[1, 2, 1] -> [1, 2, 0] ['a', 'bc', '']
PASS | file='abc' queries=[4, 1] -> [3, 0] ['abc', '']
PASS | file='abcdefg' queries=[1, 2, 1, 5, 3] -> [1, 2, 1, 3, 0] ['a', 'bc', 'd', 'efg', '']


True

**Độ phức tạp cách 3** (`n` = số ký tự yêu cầu ở một lần gọi):

- Thời gian mỗi lần gọi: `O(n)`. Mỗi ký tự được chép đúng 2 lần (file → `buf4` → `buf`); số lần gọi `read4` trong một lần `read` tối đa khoảng `n / 4 + 1`.
- Bộ nhớ: `O(1)` — chỉ có `buf4` 4 ô và hai số nguyên, **không phụ thuộc độ dài file**.

So với cách 2: không đọc trước gì cả ("lazy" — lười, cần tới đâu đọc tới đó), và không bao giờ giữ quá 4 ký tự.

## 8. Cách 4 — Dùng `collections.deque` làm hàng đợi phần dư

### 8.1. `deque` là gì?

Phần dư có tính chất **"vào trước, ra trước"** (FIFO — First In, First Out): ký tự nào `read4` trả về trước thì phải giao cho người gọi trước. Cấu trúc dữ liệu tự nhiên cho FIFO là **hàng đợi** (queue).

Trong Python, hàng đợi chuẩn là `collections.deque` (đọc là "đéc", viết tắt *double-ended queue* — hàng đợi hai đầu):

- `q = deque()` — tạo hàng đợi rỗng.
- `q.append(x)` — thêm `x` vào **cuối**.
- `q.extend(list)` — thêm nhiều phần tử vào cuối, giữ thứ tự.
- `q.popleft()` — lấy và xóa phần tử ở **đầu**, `O(1)`.
- `len(q)` / `if q:` — kiểm tra còn phần tử không.

Vì sao không dùng list với `list.pop(0)`? Vì `pop(0)` phải dời toàn bộ phần tử còn lại sang trái một ô, tốn `O(k)` với `k` là độ dài list. `deque.popleft()` chỉ tốn `O(1)`.

In [18]:
from collections import deque

q = deque()
q.extend(['a', 'b', 'c', 'd'])     # read4 trả "abcd" -> cho cả 4 vào hàng đợi
print("sau extend :", q)
print("popleft ->", q.popleft())   # giao 'a'
print("còn lại    :", q, "len =", len(q))
q.append('e')
print("append 'e' :", q)
print("rỗng chưa? :", not q)

sau extend : deque(['a', 'b', 'c', 'd'])
popleft -> a
còn lại    : deque(['b', 'c', 'd']) len = 3
append 'e' : deque(['b', 'c', 'd', 'e'])
rỗng chưa? : False


### 8.2. Ý tưởng cách 4

Thay bộ ba `buf4 / buf_ptr / buf_cnt` bằng một hàng đợi `self.queue`:

- Mỗi vòng, nếu `self.queue` rỗng → gọi `read4`, đẩy `cnt` ký tự có nghĩa vào hàng đợi. Nếu `cnt == 0` → hết file, dừng.
- Giao `self.queue.popleft()` vào `buf[count]`.

Hàng đợi tự lo việc "còn dư bao nhiêu, đang ở đâu", nên code dễ đọc hơn — đổi lại là phụ thuộc một cấu trúc dữ liệu của thư viện. Vì chỉ nạp khi hàng đợi **rỗng**, hàng đợi không bao giờ chứa quá 4 phần tử.

In [19]:
class SolutionDeque:
    def __init__(self):
        self.queue = deque()     # phần dư chưa giao, theo đúng thứ tự file
        self.buf4 = [''] * 4

    def read(self, buf: List[str], n: int) -> int:
        count = 0
        while count < n:
            if not self.queue:                     # hết phần dư -> nạp thêm
                cnt = read4(self.buf4)
                if cnt == 0:                       # hết file
                    break
                self.queue.extend(self.buf4[:cnt])
            buf[count] = self.queue.popleft()
            count += 1
        return count


check_queries(SolutionDeque, "abc", [1, 2, 1])
check_queries(SolutionDeque, "abc", [4, 1])
check_queries(SolutionDeque, "abcdefg", [1, 2, 1, 5, 3])

PASS | file='abc' queries=[1, 2, 1] -> [1, 2, 0] ['a', 'bc', '']
PASS | file='abc' queries=[4, 1] -> [3, 0] ['abc', '']
PASS | file='abcdefg' queries=[1, 2, 1, 5, 3] -> [1, 2, 1, 3, 0] ['a', 'bc', 'd', 'efg', '']


True

### Trace cách 4 với cùng input `"abcdefg"`, `[1, 2, 1, 5, 3]`

Lần này ta in nội dung hàng đợi sau mỗi lần gọi `read` — nó chính là "còn dư sau" trong bảng ở mục 7.

In [20]:
def trace_deque(content, queries):
    set_file(content)
    sol = SolutionDeque()
    for n in queries:
        before = ''.join(sol.queue)
        calls_before = _current_file.read4_calls
        buf = [''] * n
        cnt = sol.read(buf, n)
        print(f"read({n}): queue trước={before!r:7} | read4 gọi thêm "
              f"{_current_file.read4_calls - calls_before} lần | trả {cnt}, "
              f"buf={''.join(buf[:cnt])!r:6} | queue sau={''.join(sol.queue)!r}")

trace_deque("abcdefg", [1, 2, 1, 5, 3])

read(1): queue trước=''      | read4 gọi thêm 1 lần | trả 1, buf='a'    | queue sau='bcd'
read(2): queue trước='bcd'   | read4 gọi thêm 0 lần | trả 2, buf='bc'   | queue sau='d'
read(1): queue trước='d'     | read4 gọi thêm 0 lần | trả 1, buf='d'    | queue sau=''
read(5): queue trước=''      | read4 gọi thêm 2 lần | trả 3, buf='efg'  | queue sau=''
read(3): queue trước=''      | read4 gọi thêm 1 lần | trả 0, buf=''     | queue sau=''


Khớp hoàn toàn với bảng của cách 3: phần dư `"bcd"` → `"d"` → `""`, và `read4` được gọi `1, 0, 0, 2, 1` lần.

**Độ phức tạp cách 4:**

- Thời gian mỗi lần gọi: `O(n)` (`extend` tối đa 4 phần tử, `popleft` là `O(1)`).
- Bộ nhớ: `O(1)` — hàng đợi tối đa 4 phần tử.

## 9. Code nộp LeetCode — cách 3 (bộ đệm 4 ô)

Khi nộp:

- **Không** định nghĩa `read4` — LeetCode đã có sẵn hàm toàn cục `read4`. (Trong notebook, `read4` của ta ở mục 2.5 đóng vai trò đó.)
- LeetCode đã `from typing import List` sẵn; trong notebook ta đã import ở cell đầu.
- Toàn bộ trạng thái nằm trong `__init__` để mỗi test case có `Solution()` mới, sạch.

In [21]:
class Solution:
    def __init__(self):
        self.buf4 = [''] * 4
        self.buf_ptr = 0
        self.buf_cnt = 0

    def read(self, buf: List[str], n: int) -> int:
        idx = 0
        while idx < n:
            if self.buf_ptr == self.buf_cnt:
                self.buf_cnt = read4(self.buf4)
                self.buf_ptr = 0
                if self.buf_cnt == 0:
                    break
            buf[idx] = self.buf4[self.buf_ptr]
            idx += 1
            self.buf_ptr += 1
        return idx

### Bộ test cho `Solution`

Các nhóm test:

1. Hai ví dụ của đề.
2. File 1 ký tự.
3. Độ dài file **là bội của 4** (`"abcd"`, `"abcdefgh"`) — kiểm tra lần `read4` trả `0` ngay sau một khối đầy.
4. Đọc từng ký tự một (`n = 1` liên tục), đọc vượt quá file, gọi sau khi file hết.
5. Biên của constraints: file 500 ký tự, `n = 500`.

In [22]:
tests = [
    ("abc", [1, 2, 1], [1, 2, 0]),
    ("abc", [4, 1], [3, 0]),
    ("a", [1, 1], [1, 0]),
    ("abcd", [4, 1], [4, 0]),
    ("abcdefgh", [3, 3, 3], [3, 3, 2]),
    ("abcdefgh", [4, 4, 1], [4, 4, 0]),
    ("abcdefghijk", [1] * 12, [1] * 11 + [0]),
    ("leetcode", [5, 2, 2], [5, 2, 1]),
    ("ab12XY", [2, 10, 3], [2, 4, 0]),
    ("z" * 500, [500, 500], [500, 0]),
]

for content, queries, expected_counts in tests:
    counts, texts = run_queries(Solution, content, queries)
    assert counts == expected_counts, (content, queries, counts)
    assert (counts, texts) == expected_output(content, queries)
    shown = content if len(content) <= 12 else content[:6] + "..."
    print(f"PASS | file={shown!r:16} queries={str(queries)[:22]:22} -> {str(counts)[:30]}")

print("Tất cả test của Solution đều PASS!")

PASS | file='abc'            queries=[1, 2, 1]              -> [1, 2, 0]
PASS | file='abc'            queries=[4, 1]                 -> [3, 0]
PASS | file='a'              queries=[1, 1]                 -> [1, 0]
PASS | file='abcd'           queries=[4, 1]                 -> [4, 0]
PASS | file='abcdefgh'       queries=[3, 3, 3]              -> [3, 3, 2]
PASS | file='abcdefgh'       queries=[4, 4, 1]              -> [4, 4, 0]
PASS | file='abcdefghijk'    queries=[1, 1, 1, 1, 1, 1, 1,  -> [1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
PASS | file='leetcode'       queries=[5, 2, 2]              -> [5, 2, 1]
PASS | file='ab12XY'         queries=[2, 10, 3]             -> [2, 4, 0]
PASS | file='zzzzzz...'      queries=[500, 500]             -> [500, 0]
Tất cả test của Solution đều PASS!


### Bản nộp thay thế — cách 4 (`deque`)

Nếu thích hàng đợi, bản dưới cũng nộp được (khi nộp, đổi tên `SolutionDequeSubmit` thành `Solution` và thêm `from collections import deque` ở đầu).

In [23]:
from collections import deque


class SolutionDequeSubmit:
    def __init__(self):
        self.queue = deque()
        self.buf4 = [''] * 4

    def read(self, buf: List[str], n: int) -> int:
        count = 0
        while count < n:
            if not self.queue:
                cnt = read4(self.buf4)
                if cnt == 0:
                    break
                self.queue.extend(self.buf4[:cnt])
            buf[count] = self.queue.popleft()
            count += 1
        return count


for content, queries, expected_counts in tests:
    assert run_queries(SolutionDequeSubmit, content, queries)[0] == expected_counts
print("SolutionDequeSubmit PASS toàn bộ test!")

SolutionDequeSubmit PASS toàn bộ test!


### Kiểm tra chéo ngẫu nhiên

Sinh 500 test ngẫu nhiên nhỏ (file 1–30 ký tự chữ/số, 1–10 lần gọi, mỗi `n` từ 1 đến 15), so mọi cách với `expected_output` (cắt chuỗi trực tiếp). Đồng thời đếm xem cách ngây thơ sai bao nhiêu lần.

In [24]:
import random
import string

random.seed(158)
alphabet = string.ascii_letters + string.digits
correct_classes = [SolutionReadAll, SolutionBuffer, SolutionBufferChunk,
                   SolutionDeque, Solution, SolutionDequeSubmit]
naive_fail = 0

for _ in range(500):
    content = ''.join(random.choice(alphabet) for _ in range(random.randint(1, 30)))
    queries = [random.randint(1, 15) for _ in range(random.randint(1, 10))]
    expected = expected_output(content, queries)
    for cls in correct_classes:
        assert run_queries(cls, content, queries) == expected, (cls.__name__, content, queries)
    if run_queries(SolutionNaive, content, queries) != expected:
        naive_fail += 1

print("6 lớp đúng: khớp đáp án chuẩn trên 500/500 test ngẫu nhiên")
print("Cách ngây thơ (bản I gọi nhiều lần) sai", naive_fail, "/ 500 test")

6 lớp đúng: khớp đáp án chuẩn trên 500/500 test ngẫu nhiên
Cách ngây thơ (bản I gọi nhiều lần) sai 292 / 500 test


## 10. Vì sao cách 3 đúng?

Gọi "**đã đọc**" là đoạn đầu `file[0 : fp]` mà `read4` đã lấy ra (fp là con trỏ file của `read4`), và "**đã giao**" là phép nối mọi ký tự mà các lần `read` trước đã ghi vào `buf`.

**Bất biến (invariant)** — đúng trước và sau mỗi lần gọi `read`, và sau mỗi vòng lặp:

> `đã giao` + `buf4[buf_ptr : buf_cnt]` == `đã đọc`

Nói bằng lời: mọi ký tự `read4` đã lấy ra thì **hoặc đã giao, hoặc đang nằm trong phần dư**, đúng thứ tự — **không ký tự nào bị mất, không ký tự nào bị lặp**.

Vì sao bất biến được giữ?

- **Ban đầu**: chưa giao gì, phần dư rỗng (`0:0`), chưa đọc gì → `"" + "" == ""`.
- **Bước giao** (`buf[idx] = buf4[buf_ptr]; buf_ptr += 1`): một ký tự chuyển từ **đầu** phần dư sang **cuối** "đã giao" → tổng không đổi, thứ tự không đổi.
- **Bước nạp** chỉ xảy ra khi phần dư **rỗng** (`buf_ptr == buf_cnt`): `read4` lấy `cnt` ký tự kế tiếp của file, ta đặt chúng làm phần dư mới (`buf_ptr = 0, buf_cnt = cnt`) → "đã đọc" dài thêm đúng những ký tự đó, vế trái cũng dài thêm đúng những ký tự đó.

Hệ quả: "đã giao" luôn là một **đoạn đầu** của file, nên mỗi lần `read` giao đúng các ký tự kế tiếp.

**Đủ `n` hoặc hết file:** vòng lặp chỉ dừng khi `idx == n` (đủ) hoặc khi phần dư rỗng **và** `read4` trả `0` (file hết thật sự, vì `read4` chỉ trả `0` ở cuối file). Vậy `read` trả `min(n, số ký tự chưa giao còn lại)` — đúng yêu cầu.

**Phần dư ≤ 3 sau mỗi lần gọi:** nạp chỉ xảy ra khi còn cần ký tự (`idx < n`), và ngay sau khi nạp ta giao ít nhất 1 ký tự.

Cell dưới đây **kiểm tra bất biến bằng code** trên 300 test ngẫu nhiên.

In [25]:
def check_invariant(content, queries):
    file = set_file(content)
    sol = SolutionBuffer()
    delivered = ""
    for n in queries:
        buf = [''] * n
        cnt = sol.read(buf, n)
        delivered += ''.join(buf[:cnt])
        leftover = ''.join(sol.buf4[sol.buf_ptr:sol.buf_cnt])
        assert delivered + leftover == content[:file.pos]   # bất biến chính
        assert len(leftover) <= 3                            # phần dư tối đa 3
        assert content.startswith(delivered)                 # đã giao = đoạn đầu file


random.seed(1580)
for _ in range(300):
    content = ''.join(random.choice(alphabet) for _ in range(random.randint(1, 40)))
    queries = [random.randint(1, 12) for _ in range(random.randint(1, 10))]
    check_invariant(content, queries)

print("Bất biến đúng trên 300 test ngẫu nhiên")

Bất biến đúng trên 300 test ngẫu nhiên


## 11. Độ phức tạp và so sánh các cách

Ký hiệu: `L` = độ dài file, `n` = số ký tự yêu cầu trong **một** lần gọi `read`.

| Cách | Ý tưởng | Thời gian / lần gọi | Bộ nhớ thêm | Đúng? | Nhận xét |
|---|---|---|---|---|---|
| Bản I (#157) | Gọi `read4`, lấy `min`, vứt phần dư | `O(n)` | `O(1)` | Chỉ khi gọi 1 lần | Nền tảng để hiểu bài |
| 1. Ngây thơ | Dùng y nguyên bản I cho mỗi lần gọi | `O(n)` | `O(1)` | **Sai** | Mất phần dư → lần sau đọc hụt |
| 2. Đọc hết file | Lần đầu gom cả file vào `self.data` | Lần đầu `O(L)`, sau `O(n)` | `O(L)` | Đúng | Trái tinh thần stream, hỏng với file khổng lồ / stream vô hạn |
| 3. Bộ đệm 4 ô | `buf4` + `buf_ptr` + `buf_cnt` | `O(n)` | `O(1)` | Đúng | **Cách chuẩn phỏng vấn**, giống `BufferedReader` thật |
| 3b. Chép theo khối | Như 3, chép `take` ký tự bằng slice | `O(n)` | `O(1)` | Đúng | Ít vòng lặp hơn, dễ sai chỉ số hơn |
| 4. `deque` | Hàng đợi FIFO chứa phần dư | `O(n)` | `O(1)` (≤ 4 phần tử) | Đúng | Dễ đọc, dựa vào thư viện |

Tổng chi phí cho cả test case (Q lần gọi, tổng số ký tự yêu cầu `N = n1 + ... + nQ`): cách 3 và 4 là `O(N + Q)` thời gian; số lần gọi `read4` không vượt quá khoảng `L / 4 + Q`.

**Nên nhớ cách nào?** Cách 3. Nó không cần thư viện, chạy được y hệt trong C/C++/Java, và chính là cách các thư viện I/O thật (ví dụ `java.io.BufferedReader`, `io.BufferedReader` của Python, `FILE*` trong C) hoạt động: đọc theo khối lớn từ nguồn chậm, giữ phần dư trong bộ đệm, giao từng phần nhỏ cho người dùng.

## 12. Các lỗi người mới hay gặp

### Lỗi 1: Gán lại `buf` thay vì ghi vào `buf`

Sai:

```python
def read(self, buf, n):
    result = []
    ...                      # gom ký tự vào result
    buf = result             # chỉ đổi tên cục bộ!
    return len(result)
```

Giá trị trả về đúng nhưng `buf` của người gọi **trống trơn** — LeetCode sẽ báo sai nội dung. Nhớ lại mục 2.2: phải **mutate** (`buf[i] = ...` hoặc `buf[i:j] = ...`).

In [26]:
class SolutionRebindBug:
    def __init__(self):
        self.inner = SolutionBuffer()

    def read(self, buf, n):
        tmp = [''] * n
        cnt = self.inner.read(tmp, n)
        buf = tmp                       # SAI: không ghi vào buf của người gọi
        return cnt


print("sai :", run_queries(SolutionRebindBug, "abc", [1, 2, 1]))
print("đúng:", run_queries(Solution, "abc", [1, 2, 1]))

sai : ([1, 2, 0], ['', '', ''])
đúng: ([1, 2, 0], ['a', 'bc', ''])


Số trả về `[1, 2, 0]` giống hệt, nhưng nội dung `buf` là `['', '', '']`. Đây là lỗi "nhìn output số thì tưởng đúng". Sửa: `buf[:cnt] = tmp[:cnt]` hoặc ghi trực tiếp `buf[idx] = ...` như cách 3.

### Lỗi 2: Khai báo trạng thái là *class attribute* (quên "RESET")

Sai:

```python
class Solution:
    queue = deque()            # dùng chung cho MỌI test case!
    def read(self, buf, n): ...
```

LeetCode chạy nhiều test case liên tiếp trong cùng tiến trình. Phần dư của test trước còn kẹt trong `Solution.queue` và bị giao nhầm cho test sau.

In [27]:
class SolutionSharedBug:
    queue = deque()                     # SAI: class attribute
    buf4 = [''] * 4

    def read(self, buf, n):
        count = 0
        while count < n:
            if not self.queue:
                cnt = read4(self.buf4)
                if cnt == 0:
                    break
                self.queue.extend(self.buf4[:cnt])
            buf[count] = self.queue.popleft()
            count += 1
        return count


print("test case 1:", run_queries(SolutionSharedBug, "abc", [1]))   # để lại "bc" trong queue chung
print("test case 2:", run_queries(SolutionSharedBug, "xyz", [3]), "<- mong đợi ([3], ['xyz'])")

test case 1: ([1], ['a'])
test case 2: ([3], ['bcx']) <- mong đợi ([3], ['xyz'])


Test case 2 nhận `"bcx"` — hai ký tự `"bc"` là rác của test case 1! Khi chạy riêng từng test thì đúng, nộp lên lại sai một cách khó hiểu. Sửa: mọi trạng thái khởi tạo trong `__init__` bằng `self.xxx = ...`.

### Lỗi 3: Quên đặt lại `buf_ptr = 0` khi nạp bộ đệm

Sai:

```python
if self.buf_ptr == self.buf_cnt:
    self.buf_cnt = read4(self.buf4)
    # quên: self.buf_ptr = 0
    if self.buf_cnt == 0:
        break
```

Sau lần nạp thứ hai, `buf_ptr` vẫn là `4` → đọc `buf4[4]` → vượt chỉ số (hoặc ở ngôn ngữ khác thì đọc rác).

In [28]:
class SolutionNoResetBug:
    def __init__(self):
        self.buf4 = [''] * 4
        self.buf_ptr = 0
        self.buf_cnt = 0

    def read(self, buf, n):
        idx = 0
        while idx < n:
            if self.buf_ptr == self.buf_cnt:
                self.buf_cnt = read4(self.buf4)   # SAI: thiếu self.buf_ptr = 0
                if self.buf_cnt == 0:
                    break
            buf[idx] = self.buf4[self.buf_ptr]
            idx += 1
            self.buf_ptr += 1
        return idx


try:
    run_queries(SolutionNoResetBug, "abcdef", [6])
except IndexError as e:
    print("IndexError:", e, "-> buf_ptr vẫn = 4 sau khi nạp lại")
print("đúng:", run_queries(Solution, "abcdef", [6]))

IndexError: list index out of range -> buf_ptr vẫn = 4 sau khi nạp lại
đúng: ([6], ['abcdef'])


### Lỗi 4: Chép cả `cnt` ký tự thay vì `min(cnt, n - total)`

Sai (trong cách kiểu bản I):

```python
cnt = read4(buf4)
for i in range(cnt):          # chép cả 4 dù chỉ cần 1
    buf[total + i] = buf4[i]
total += cnt
```

Hai hậu quả: (1) ghi **vượt** `n` ô của `buf` — trong Python là `IndexError` nếu `buf` đúng `n` ô, trong C/C++ là ghi đè bộ nhớ (undefined behavior); (2) trả về số lớn hơn `n`.

In [29]:
def read_overflow_bug(buf, n):
    buf4, total = [''] * 4, 0
    while total < n:
        cnt = read4(buf4)
        if cnt == 0:
            break
        for i in range(cnt):            # SAI: thiếu min(cnt, n - total)
            buf[total + i] = buf4[i]
        total += cnt
    return total


set_file("abc")
try:
    read_overflow_bug([''] * 1, 1)
except IndexError as e:
    print("IndexError:", e, "-> cố ghi buf[1] trong khi buf chỉ có 1 ô")

IndexError: list assignment index out of range -> cố ghi buf[1] trong khi buf chỉ có 1 ô


### Lỗi 5: Quên `break` khi `read4` trả `0`

Với vòng lặp kiểu bản I, nếu quên kiểm tra `cnt == 0` thì khi file hết, `take = 0`, `total` không tăng, điều kiện `total < n` mãi đúng → **vòng lặp vô hạn** (LeetCode báo *Time Limit Exceeded*).

Cell dưới mô phỏng lỗi đó nhưng có "cầu chì" giới hạn 20 vòng để notebook không bị treo.

In [30]:
def read_no_break_bug(buf, n, max_loops=20):
    buf4, total, loops = [''] * 4, 0, 0
    while total < n:
        loops += 1
        if loops > max_loops:
            return f"vượt {max_loops} vòng -> vòng lặp không dừng!"
        cnt = read4(buf4)                 # SAI: không có  if cnt == 0: break
        take = min(cnt, n - total)
        for i in range(take):
            buf[total + i] = buf4[i]
        total += take
    return total


set_file("abc")
print(read_no_break_bug([''] * 10, 10))

vượt 20 vòng -> vòng lặp không dừng!


### Bảng tóm tắt lỗi

| Lỗi | Triệu chứng | Sửa |
|---|---|---|
| Dùng nguyên bản I | Lần gọi sau đọc hụt (`[1,0,0]` thay vì `[1,2,0]`) | Cất phần dư vào `self.*` |
| `buf = ...` | Số trả về đúng, nội dung `buf` rỗng | Ghi từng ô `buf[i] = ...` |
| Class attribute | Chạy riêng đúng, nộp lên sai | Khởi tạo trong `__init__` |
| Quên `buf_ptr = 0` | `IndexError` / đọc rác | Đặt lại ngay sau `read4` |
| Không dùng `min` | Ghi vượt `buf`, trả > `n` | `take = min(cnt, n - total)` |
| Quên `break` khi `cnt == 0` | Vòng lặp vô hạn (TLE) | `if cnt == 0: break` |
| Dùng cả 4 ô `buf4` | Lẫn ký tự rác cuối file | Chỉ dùng `buf4[:cnt]` |

## 13. Mẹo nhận dạng pattern, biến thể và bài liên quan

### Dấu hiệu nhận dạng

- Nguồn dữ liệu chỉ cho lấy theo **khối cố định** (4, 512, 4096 byte...) nhưng người dùng muốn số lượng **tùy ý**.
- Hàm/method bị **gọi nhiều lần** và nguồn dữ liệu **không quay lui** được.
- Đề nhắc tới "stream", "iterator", "API có sẵn", "con trỏ riêng".

→ Nghĩ ngay tới **bộ đệm nội bộ + con trỏ** (buffered reader): *nạp khi rỗng, giao từ bộ đệm, nhớ phần dư trong object.*

### Câu thần chú

> **"Rỗng thì nạp, nạp được 0 thì dừng, còn thì giao, giao xong thì tiến con trỏ."**

### Biến thể hay gặp

- `readK` thay vì `read4`: bộ đệm `k` ô, logic giữ nguyên (Bài tập 5).
- Thêm `peek()` — xem ký tự kế tiếp mà **không** tiêu thụ: chỉ cần nạp nếu rỗng rồi trả `buf4[buf_ptr]` mà không tăng con trỏ (Bài tập 6).
- `readLine()` — đọc tới ký tự xuống dòng: cùng vòng lặp, điều kiện dừng khác.
- Đọc trực tiếp vào `buf` khi còn cần ≥ 4 ký tự và bộ đệm rỗng (bỏ qua một lần chép) — tối ưu hằng số mà các thư viện thật hay làm.

### Bài liên quan trên LeetCode

| Bài | Liên hệ |
|---|---|
| #157 Read N Characters Given Read4 | Bản I, gọi 1 lần — không cần trạng thái |
| #284 Peeking Iterator | Giữ "phần tử nhìn trước" trong object — cùng tư tưởng bộ đệm 1 ô |
| #251 Flatten 2D Vector | Iterator với hai con trỏ trạng thái |
| #173 Binary Search Tree Iterator | Iterator giữ trạng thái (stack) giữa các lần `next()` |
| #604 Design Compressed String Iterator | Giữ "ký tự hiện tại + số lần còn lại" giữa các lần gọi |
| #900 RLE Iterator | Tiêu thụ dần một nguồn theo khối, giữ phần dư của khối hiện tại |

## 14. Bài tập vận dụng

Làm theo thứ tự, từ đoán output tới mini challenge. **Đừng xem đáp án ở cuối notebook trước khi tự làm.**

### Bài 1 — Đoán output (cách ngây thơ)

Với `SolutionNaive` (dùng nguyên bản I), file `"abcdefgh"`, `queries = [3, 1, 4]`:

- Mỗi lần gọi trả về bao nhiêu? `buf` chứa gì?
- Ký tự nào bị vứt ở mỗi lần gọi?
- Đáp án đúng (của cách 3) là gì?

Hãy tự viết ra giấy trước, rồi mới xem đáp án.

### Bài 2 — Tính tay bảng trạng thái (cách 3)

Với file `"abcdefghijk"` (11 ký tự), `queries = [3, 3, 3, 3]`, hãy điền bảng:

| Lệnh | Còn dư trước | Các lần gọi `read4` (trả gì) | Trả về | `buf_ptr`, `buf_cnt` sau |
|---|---|---|---:|---|
| `read(3)` | ? | ? | ? | ? |
| `read(3)` | ? | ? | ? | ? |
| `read(3)` | ? | ? | ? | ? |
| `read(3)` | ? | ? | ? | ? |

Tổng cộng `read4` được gọi mấy lần?

### Bài 3 — Điền TODO cho cách 3

Hoàn thiện `ExerciseBufferReader` theo đúng 5 bước ở mục 7.

In [31]:
class ExerciseBufferReader:
    def __init__(self):
        # TODO 1: tạo self.buf4 (4 ô), self.buf_ptr, self.buf_cnt
        pass

    def read(self, buf: List[str], n: int) -> int:
        # TODO 2: idx = 0; lặp khi idx < n
        # TODO 3: nếu bộ đệm rỗng -> gọi read4, đặt lại buf_ptr, nếu cnt == 0 thì break
        # TODO 4: chép self.buf4[self.buf_ptr] vào buf[idx], tăng idx và buf_ptr
        # TODO 5: trả idx
        raise NotImplementedError


# Tự làm xong rồi mới mở test:
# assert run_queries(ExerciseBufferReader, "abc", [1, 2, 1]) == ([1, 2, 0], ['a', 'bc', ''])
# assert run_queries(ExerciseBufferReader, "abc", [4, 1]) == ([3, 0], ['abc', ''])
# assert run_queries(ExerciseBufferReader, "abcdefg", [1, 2, 1, 5, 3])[0] == [1, 2, 1, 3, 0]

### Bài 4 — Tự viết lại cách 4 bằng `deque`

Viết `ExerciseDequeReader` **không nhìn lại** mục 8. Yêu cầu: trạng thái trong `__init__`, chỉ nạp khi hàng đợi rỗng, chỉ đẩy `buf4[:cnt]`.

In [32]:
from collections import deque


class ExerciseDequeReader:
    def __init__(self):
        # TODO: tạo hàng đợi và buf4
        pass

    def read(self, buf: List[str], n: int) -> int:
        # TODO
        raise NotImplementedError


# assert run_queries(ExerciseDequeReader, "abc", [1, 2, 1]) == ([1, 2, 0], ['a', 'bc', ''])
# assert run_queries(ExerciseDequeReader, "abcdefgh", [2, 2, 4]) == ([2, 2, 4], ['ab', 'cd', 'efgh'])

### Bài 5 — Biến thể: `readK` thay vì `read4`

Giả sử API có sẵn là `readk(bufk)` đọc tối đa **`k`** ký tự (k không nhất thiết là 4). Hàm `make_readk(content, k)` dưới đây tạo sẵn một `readk` như vậy cho bạn (đã viết xong, không phải TODO).

Hãy viết class `ReaderK(readk, k)` có method `read(buf, n)` hoạt động như bài gốc nhưng dùng `readk` và bộ đệm `k` ô.

In [33]:
def make_readk(content: str, k: int):
    # Đã viết sẵn: tạo một hàm readk có con trỏ file riêng, mỗi lần đọc tối đa k ký tự.
    state = {"pos": 0}

    def readk(bufk: List[str]) -> int:
        count = 0
        while count < k and state["pos"] < len(content):
            bufk[count] = content[state["pos"]]
            state["pos"] += 1
            count += 1
        return count

    return readk


def run_reader_k(content, k, queries, reader_cls):
    reader = reader_cls(make_readk(content, k), k)
    counts, texts = [], []
    for n in queries:
        buf = [''] * n
        cnt = reader.read(buf, n)
        counts.append(cnt)
        texts.append(''.join(buf[:cnt]))
    return counts, texts


class ExerciseReaderK:
    def __init__(self, readk, k: int):
        # TODO: lưu readk, k; tạo bộ đệm k ô, buf_ptr, buf_cnt
        pass

    def read(self, buf: List[str], n: int) -> int:
        # TODO: giống cách 3, thay read4 bằng self.readk
        raise NotImplementedError


# assert run_reader_k("abcdefg", 3, [1, 2, 1, 5], ExerciseReaderK) == expected_output("abcdefg", [1, 2, 1, 5])
# assert run_reader_k("abcdefg", 1, [2, 9], ExerciseReaderK) == expected_output("abcdefg", [2, 9])

### Bài 6 — Mini challenge: thêm `peek()`

Viết class `ExercisePeekReader` (dùng hàm toàn cục `read4`) có hai method dùng **chung** trạng thái:

- `read(buf, n) -> int`: như bài gốc.
- `peek() -> str`: trả về ký tự **kế tiếp** sẽ được đọc, **không tiêu thụ** nó; trả `''` nếu file đã hết.

Ví dụ với file `"abc"`:

```text
peek()      -> 'a'
peek()      -> 'a'      (gọi lại vẫn là 'a')
read(buf,2) -> 2, "ab"
peek()      -> 'c'
read(buf,5) -> 1, "c"
peek()      -> ''
```

Gợi ý cấu trúc: viết một method phụ `_ensure()` "nạp nếu rỗng, trả `True` nếu còn ký tự".

In [34]:
class ExercisePeekReader:
    def __init__(self):
        # TODO
        pass

    def _ensure(self) -> bool:
        # TODO: nếu bộ đệm rỗng thì nạp; trả True nếu còn ít nhất 1 ký tự
        raise NotImplementedError

    def peek(self) -> str:
        # TODO
        raise NotImplementedError

    def read(self, buf: List[str], n: int) -> int:
        # TODO
        raise NotImplementedError


# set_file("abc"); r = ExercisePeekReader()
# assert r.peek() == 'a' and r.peek() == 'a'
# buf = [''] * 2; assert r.read(buf, 2) == 2 and buf == ['a', 'b']
# assert r.peek() == 'c'
# buf = [''] * 5; assert r.read(buf, 5) == 1 and buf[0] == 'c'
# assert r.peek() == ''

## 15. Gợi ý theo từng mức

**Bài 1**
- Mức 1: mỗi lần gọi `SolutionNaive.read` tạo `buf4` mới và gọi `read4` **ít nhất một lần**.
- Mức 2: `read(3)` gọi `read4` 1 lần → nhận `"abcd"`; lấy 3, vứt 1. Con trỏ file giờ ở `'e'`.
- Mức 3: `read(1)` nhận `"efgh"`, lấy `'e'`, vứt `"fgh"`. `read(4)` gặp file hết.

**Bài 2**
- Mức 1: bắt đầu với `buf_ptr = buf_cnt = 0`; mỗi khi `buf_ptr == buf_cnt` mà còn cần ký tự thì gọi `read4`.
- Mức 2: các khối `read4` lần lượt là `"abcd"`, `"efgh"`, `"ijk"`, rồi `""`.
- Mức 3: `read(3)` thứ hai giao `'d'` từ phần dư rồi mới nạp `"efgh"`.

**Bài 3**
- Mức 1: chép đúng 5 bước ở mục 7.
- Mức 2: điều kiện bộ đệm rỗng là `self.buf_ptr == self.buf_cnt`.
- Mức 3: thứ tự trong nhánh nạp: `self.buf_cnt = read4(self.buf4)` → `self.buf_ptr = 0` → `if self.buf_cnt == 0: break`.

**Bài 4**
- Mức 1: hàng đợi là FIFO; ký tự vào trước ra trước.
- Mức 2: `if not self.queue:` để biết hàng đợi rỗng; `self.queue.extend(self.buf4[:cnt])`.
- Mức 3: `buf[count] = self.queue.popleft()`.

**Bài 5**
- Mức 1: chỉ có hai chỗ dùng số `4` trong cách 3 — đổi chúng thành `k`.
- Mức 2: `self.bufk = [''] * k`, và gọi `self.readk(self.bufk)`.
- Mức 3: lưu hàm `readk` vào `self.readk` trong `__init__`; gọi nó như method bình thường.

**Bài 6**
- Mức 1: `peek` và `read` khác nhau đúng một điều: `peek` **không tăng** `buf_ptr`.
- Mức 2: `_ensure()`: nếu `buf_ptr == buf_cnt` thì nạp (đặt `buf_ptr = 0`); trả `self.buf_ptr < self.buf_cnt`.
- Mức 3: `read` dùng `while idx < n and self._ensure(): ...`.

## 16. Đáp án tham khảo

Nên tự làm trước rồi mới xem phần này.

### Đáp án Bài 1

| Lệnh | `read4` trả | Lấy | Vứt | Trả về | Đáp án đúng |
|---|---|---|---|---:|---|
| `read(3)` | `"abcd"` | `"abc"` | `"d"` | 3 | `"abc"` (3) |
| `read(1)` | `"efgh"` | `"e"` | `"fgh"` | 1 | `"d"` (1) |
| `read(4)` | `""` | `""` | — | 0 | `"efgh"` (4) |

Cách ngây thơ cho `[3, 1, 0]` với nội dung `"abc"`, `"e"`, `""`; đáp án đúng là `[3, 1, 4]` với `"abc"`, `"d"`, `"efgh"`.

### Đáp án Bài 2

| Lệnh | Còn dư trước | Các lần gọi `read4` (trả gì) | Trả về | `buf_ptr`, `buf_cnt` sau |
|---|---|---|---:|---|
| `read(3)` | `""` | `"abcd"` | 3 (`"abc"`) | 3, 4 |
| `read(3)` | `"d"` | `"efgh"` | 3 (`"def"`) | 2, 4 |
| `read(3)` | `"gh"` | `"ijk"` | 3 (`"ghi"`) | 1, 3 |
| `read(3)` | `"jk"` | `""` (hết file) | 2 (`"jk"`) | 0, 0 |

Tổng cộng `read4` được gọi **4 lần**. Kết quả `[3, 3, 3, 2]`. Chú ý dòng cuối: sau khi `read4` trả `0`, ta đã gán `buf_cnt = 0` và `buf_ptr = 0`.

Cell dưới kiểm chứng hai đáp án trên bằng code.

In [35]:
# Kiểm chứng Bài 1
print("Bài 1 - ngây thơ:", run_queries(SolutionNaive, "abcdefgh", [3, 1, 4]))
print("Bài 1 - đúng    :", run_queries(Solution, "abcdefgh", [3, 1, 4]))
assert run_queries(SolutionNaive, "abcdefgh", [3, 1, 4]) == ([3, 1, 0], ['abc', 'e', ''])
assert run_queries(Solution, "abcdefgh", [3, 1, 4]) == ([3, 1, 4], ['abc', 'd', 'efgh'])

# Kiểm chứng Bài 2
set_file("abcdefghijk")
sol = Solution()
rows = []
for n in [3, 3, 3, 3]:
    buf = [''] * n
    cnt = sol.read(buf, n)
    rows.append((cnt, ''.join(buf[:cnt]), sol.buf_ptr, sol.buf_cnt))
print("Bài 2:", rows, "| read4 gọi", _current_file.read4_calls, "lần")
assert rows == [(3, 'abc', 3, 4), (3, 'def', 2, 4), (3, 'ghi', 1, 3), (2, 'jk', 0, 0)]
assert _current_file.read4_calls == 4
print("Đáp án Bài 1, 2 PASS!")

Bài 1 - ngây thơ: ([3, 1, 0], ['abc', 'e', ''])
Bài 1 - đúng    : ([3, 1, 4], ['abc', 'd', 'efgh'])
Bài 2: [(3, 'abc', 3, 4), (3, 'def', 2, 4), (3, 'ghi', 1, 3), (2, 'jk', 0, 0)] | read4 gọi 4 lần
Đáp án Bài 1, 2 PASS!


In [36]:
# Đáp án Bài 3
class AnswerBufferReader:
    def __init__(self):
        self.buf4 = [''] * 4
        self.buf_ptr = 0
        self.buf_cnt = 0

    def read(self, buf: List[str], n: int) -> int:
        idx = 0
        while idx < n:
            if self.buf_ptr == self.buf_cnt:
                self.buf_cnt = read4(self.buf4)
                self.buf_ptr = 0
                if self.buf_cnt == 0:
                    break
            buf[idx] = self.buf4[self.buf_ptr]
            idx += 1
            self.buf_ptr += 1
        return idx


# Đáp án Bài 4
class AnswerDequeReader:
    def __init__(self):
        self.queue = deque()
        self.buf4 = [''] * 4

    def read(self, buf: List[str], n: int) -> int:
        count = 0
        while count < n:
            if not self.queue:
                cnt = read4(self.buf4)
                if cnt == 0:
                    break
                self.queue.extend(self.buf4[:cnt])
            buf[count] = self.queue.popleft()
            count += 1
        return count


for cls in [AnswerBufferReader, AnswerDequeReader]:
    assert run_queries(cls, "abc", [1, 2, 1]) == ([1, 2, 0], ['a', 'bc', ''])
    assert run_queries(cls, "abc", [4, 1]) == ([3, 0], ['abc', ''])
    assert run_queries(cls, "abcdefg", [1, 2, 1, 5, 3])[0] == [1, 2, 1, 3, 0]
    assert run_queries(cls, "abcdefgh", [2, 2, 4]) == ([2, 2, 4], ['ab', 'cd', 'efgh'])
print("Đáp án Bài 3, 4 PASS!")

Đáp án Bài 3, 4 PASS!


In [37]:
# Đáp án Bài 5
class AnswerReaderK:
    def __init__(self, readk, k: int):
        self.readk = readk
        self.k = k
        self.bufk = [''] * k
        self.buf_ptr = 0
        self.buf_cnt = 0

    def read(self, buf: List[str], n: int) -> int:
        idx = 0
        while idx < n:
            if self.buf_ptr == self.buf_cnt:
                self.buf_cnt = self.readk(self.bufk)
                self.buf_ptr = 0
                if self.buf_cnt == 0:
                    break
            buf[idx] = self.bufk[self.buf_ptr]
            idx += 1
            self.buf_ptr += 1
        return idx


# Đáp án Bài 6
class AnswerPeekReader:
    def __init__(self):
        self.buf4 = [''] * 4
        self.buf_ptr = 0
        self.buf_cnt = 0

    def _ensure(self) -> bool:
        if self.buf_ptr == self.buf_cnt:
            self.buf_cnt = read4(self.buf4)
            self.buf_ptr = 0
        return self.buf_ptr < self.buf_cnt

    def peek(self) -> str:
        return self.buf4[self.buf_ptr] if self._ensure() else ''

    def read(self, buf: List[str], n: int) -> int:
        idx = 0
        while idx < n and self._ensure():
            buf[idx] = self.buf4[self.buf_ptr]
            idx += 1
            self.buf_ptr += 1
        return idx


assert run_reader_k("abcdefg", 3, [1, 2, 1, 5], AnswerReaderK) == expected_output("abcdefg", [1, 2, 1, 5])
assert run_reader_k("abcdefg", 1, [2, 9], AnswerReaderK) == expected_output("abcdefg", [2, 9])
random.seed(5)
for _ in range(200):
    content = ''.join(random.choice(alphabet) for _ in range(random.randint(1, 25)))
    k = random.randint(1, 6)
    queries = [random.randint(1, 10) for _ in range(random.randint(1, 8))]
    assert run_reader_k(content, k, queries, AnswerReaderK) == expected_output(content, queries)

set_file("abc"); r = AnswerPeekReader()
assert r.peek() == 'a' and r.peek() == 'a'
buf = [''] * 2; assert r.read(buf, 2) == 2 and buf == ['a', 'b']
assert r.peek() == 'c'
buf = [''] * 5; assert r.read(buf, 5) == 1 and buf[0] == 'c'
assert r.peek() == ''
for content, queries, expected_counts in tests:       # read của PeekReader vẫn đúng như bài gốc
    assert run_queries(AnswerPeekReader, content, queries)[0] == expected_counts
print("Đáp án Bài 5, 6 PASS!")

Đáp án Bài 5, 6 PASS!


## 17. Template cuối cùng nên nhớ

```python
class Solution:
    def __init__(self):
        self.buf4 = [''] * 4      # bộ đệm nội bộ
        self.buf_ptr = 0          # ô kế tiếp chưa giao
        self.buf_cnt = 0          # số ô có nghĩa

    def read(self, buf: List[str], n: int) -> int:
        idx = 0
        while idx < n:
            if self.buf_ptr == self.buf_cnt:          # rỗng thì nạp
                self.buf_cnt = read4(self.buf4)
                self.buf_ptr = 0
                if self.buf_cnt == 0:                 # nạp được 0 thì dừng
                    break
            buf[idx] = self.buf4[self.buf_ptr]        # còn thì giao
            idx += 1
            self.buf_ptr += 1                         # giao xong thì tiến con trỏ
        return idx
```

Công thức nhớ nhanh:

- **Trạng thái** = `buf4` + `buf_ptr` + `buf_cnt`, khởi tạo trong `__init__`.
- **Rỗng** ⇔ `buf_ptr == buf_cnt`. **Hết file** ⇔ vừa nạp mà `buf_cnt == 0`.
- **Phần dư** = `buf4[buf_ptr:buf_cnt]`, tối đa 3 ký tự, luôn được giao trước khi gọi `read4`.

## 18. Checklist tự kiểm tra trước khi submit

- Mình có **giữ phần dư** giữa các lần gọi (trong `self.*`) chưa, hay vẫn vứt như bản I?
- Mọi trạng thái có nằm trong `__init__` (không phải class attribute) chưa?
- Mình có **ghi vào `buf`** (`buf[i] = ...`) chứ không gán lại `buf = ...` chưa?
- Khi nạp lại, mình có đặt `buf_ptr = 0` chưa?
- Khi `read4` trả `0`, mình có `break` chưa (tránh vòng lặp vô hạn / đọc rác)?
- Mình chỉ dùng `buf4[:cnt]` (không dùng cả 4 ô) chưa?
- Mình có lấy `min(...)` để không ghi quá `n` ô chưa?
- Đã test: hai ví dụ của đề (`"abc"` với `[1,2,1]` → `[1,2,0]`, `"abc"` với `[4,1]` → `[3,0]`), file có độ dài là bội của 4, đọc từng ký tự, gọi sau khi file hết chưa?
- Mình **không** định nghĩa lại `read4` trong code nộp chứ?